In [3]:
import pandas as pd


estimation_method_abbr = {
    "Linguistic Confidence": "Linguistic Conf.",
    "Semantic Uncertainty": "Semantic Unc.",
    "Self-Evaluation": "Self-Eval.",
    "Token Probability": "Token Prob."
}

def format_model(model):
    model_map = {
        "LLAMA-3.1-8B-INSTRUCT": "\\begin{tabular}{@{}c@{}}LLAMA-3.1-\\\\ 8B-Instruct\\end{tabular}",
        "META-LLAMA-3-8B-INSTRUCT": "\\begin{tabular}{@{}c@{}}LLAMA-3-\\\\ 8B-Instruct\\end{tabular}",
        "MISTRAL-7B-INSTRUCT-V0.3": "\\begin{tabular}{@{}c@{}}MISTRAL-7B-\\\\ Instruct-V0.3\\end{tabular}",
        "QWEN2.5-7B-INSTRUCT": "\\begin{tabular}{@{}c@{}}QWEN-2.5-\\\\ 7B-Instruct\\end{tabular}",
        "QWEN3-8B": "Qwen3-8B",
        "GPT-OSS-20B": "GPT-OSS-20B" 
    }
    formatted_name = model_map.get(model)
    if formatted_name:
        return formatted_name
    return model

def generate_latex_table(csv_files, table_type='raw'):
    """
    csv_files: dict mapping Dataset Name to Filename
    table_type: 'raw' or 'pt'
    """
    all_data = []
    
    # Load and label datasets
    # Ordered according to your requirement: Trivia QA -> MMLU -> SQuAD 2.0
    order = ["Trivia QA", "MMLU", "SQuAD 2.0"]
    
    for dataset_name in order:
        filename = csv_files.get(dataset_name)
        if filename:
            try:
                temp_df = pd.read_csv(filename)
                temp_df['Dataset'] = dataset_name
                all_data.append(temp_df)
            except FileNotFoundError:
                print(f"Warning: {filename} not found.")

    if not all_data:
        return "No data found."

    df = pd.concat(all_data, ignore_index=True)

    # Column Mapping based on table type
    if table_type == 'raw':
        cols = {
            'ece_raw': 'raw_dECE',
            'ece_reg': 'dECE_reg',
            'ece_kl': 'dECE_kl',
            'auc_raw': 'raw_dAUROC',
            'auc_reg': 'dAUROC_reg',
            'auc_kl': 'dAUROC_kl',
            'caption': "Calibration and discrimination results for distributional post-hoc calibration. Metrics reported in distributional metrics."
        }
        calib_label = "$dECE$"
        discr_label = "$dAUROC$"
    else:
        cols = {
            'ece_raw': 'raw_dECE_pt',
            'ece_reg': 'dECE_pt_reg',
            'ece_kl': 'dECE_pt_kl',
            'auc_raw': 'raw_dAUROC_pt',
            'auc_reg': 'dAUROC_pt_reg',
            'auc_kl': 'dAUROC_pt_kl',
            'caption': "Calibration and discrimination results for distributional post-hoc calibration. Metrics reported in collapsed distributional metrics (reduced to means)."
        }
        calib_label = "$dECE_{PT}$"
        discr_label = "$dAUROC_{PT}$"

    # Start LaTeX String using longtable
    latex = [
        "\\small",
        "\\setlength{\\tabcolsep}{4pt}",
        "\\renewcommand{\\arraystretch}{1.0}",
        "",
        "\\begin{longtable}{llcccccc}",
        f"\\caption{{{cols['caption']}}}\\\\",
        "\\toprule",
        f"Model & Estimation & \\multicolumn{{3}}{{c}}{{Calibration ({calib_label})}} & \\multicolumn{{3}}{{c}}{{Discrimination ({discr_label})}} \\\\",
        "\\cmidrule(lr){3-5} \\cmidrule(lr){6-8}",
        " & Method & Uncal. & Platt + Iso. Reg. & Platt + KL & Uncal. & Platt + Iso. Reg. & Platt + KL \\\\",
        "\\midrule",
        "\\endfirsthead",
        "\\multicolumn{8}{l}{\\textit{(continued from previous page)}}\\\\",
        "\\toprule",
        f"Model & Estimation & \\multicolumn{{3}}{{c}}{{Calibration ({calib_label})}} & \\multicolumn{{3}}{{c}}{{Discrimination ({discr_label})}} \\\\",
        "\\cmidrule(lr){3-5} \\cmidrule(lr){6-8}",
        " & Method & Uncal. & Platt + Iso. Reg. & Platt + KL & Uncal. & Platt + Iso. Reg. & Platt + KL \\\\",
        "\\midrule",
        "\\endhead",
        # "\\midrule \\multicolumn{8}{r}{\\textit{(continued on next page)}}\\\\",
        "\\endfoot",
        "\\bottomrule",
        "\\endlastfoot"
    ]

    for dataset in order:
        if dataset not in df['Dataset'].unique():
            continue
        
        latex.append(f"\\multicolumn{{8}}{{c}}{{\\textbf{{{dataset}}}}} \\\\")
        latex.append("\\midrule")
        
        ds_df = df[df['Dataset'] == dataset]
        for model in ds_df['Model'].unique():
            model_df = ds_df[ds_df['Model'] == model]
            n_rows = len(model_df)
            
            for i, (_, row) in enumerate(model_df.iterrows()):
                # Multirow logic for the Model column
                model_fmt = format_model(model)
                model_cell = f"\\multirow{{{n_rows}}}{{*}}{{{model_fmt}}}" if i == 0 else ""
                
                # Format numbers to 4 decimal places
                line = (
                    f"{model_cell} & {estimation_method_abbr.get(row['Estimation Method'])} & "
                    f"{row[cols['ece_raw']]:.4f} $\\rightarrow$ & {row[cols['ece_reg']]:.4f} & {row[cols['ece_kl']]:.4f} & "
                    f"{row[cols['auc_raw']]:.4f} $\\rightarrow$ & {row[cols['auc_reg']]:.4f} & {row[cols['auc_kl']]:.4f} \\\\"
                )
                latex.append(line)
            latex.append("\\midrule")

    latex.extend(["\\end{longtable}"])
    return "\n".join(latex)

# Configuration
files = {
    "Trivia QA": "calibration_table_trivia_qa.csv",
    "MMLU": "calibration_table_mmlu.csv",
    "SQuAD 2.0": "calibration_table_squadv2.csv"
}

# Generate and print tables
# print("--- RAW METRICS TABLE ---")
print(generate_latex_table(files, table_type='raw'))
print("\\clearpage")
# print("\n\n--- PT METRICS TABLE ---")
print(generate_latex_table(files, table_type='pt'))

\small
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.0}

\begin{longtable}{llcccccc}
\caption{Calibration and discrimination results for distributional post-hoc calibration. Metrics reported in distributional metrics.}\\
\toprule
Model & Estimation & \multicolumn{3}{c}{Calibration ($dECE$)} & \multicolumn{3}{c}{Discrimination ($dAUROC$)} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
 & Method & Uncal. & Platt + Iso. Reg. & Platt + KL & Uncal. & Platt + Iso. Reg. & Platt + KL \\
\midrule
\endfirsthead
\multicolumn{8}{l}{\textit{(continued from previous page)}}\\
\toprule
Model & Estimation & \multicolumn{3}{c}{Calibration ($dECE$)} & \multicolumn{3}{c}{Discrimination ($dAUROC$)} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
 & Method & Uncal. & Platt + Iso. Reg. & Platt + KL & Uncal. & Platt + Iso. Reg. & Platt + KL \\
\midrule
\endhead
\endfoot
\bottomrule
\endlastfoot
\multicolumn{8}{c}{\textbf{Trivia QA}} \\
\midrule
\multirow{4}{*}{\begin{tabular}{@{}c@{}}LLAMA-3.1-\\ 8B-Instruct